# Run_Eval -- grade the conversations, write the score lake

**This is the one place scores are produced.** Every other module in `eda_analysis` reads finished
artifacts; this notebook makes them. It walks each arm's conversations on disk, asks a JUDGE to
fill every rubric, and writes one parquet per `(judge, rep, metric, arm, model state)`:

```
data/eval_scores/judge=<tag>/rep=<r>/metric=<M>/<EXPERIMENT_NAME>/model_iter_<N>.parquet
```

96 rows each -- one per persona -- with `persona_id` in every row. That column is the only valid
pairing key downstream; never pair on file order or row order.

### Re-running costs nothing

A partition is **done** when its parquet exists and holds a row for every conversation currently on
disk for that model state, and `discover_scorable` skips exactly those. So a second pass over a
finished lake issues **zero** grader calls, an interrupted pass resumes at the next missing file,
and a partition that errored simply stays in the plan until it succeeds. Resume granularity is one
whole parquet: a partially-scored state is re-scored in full, which is the price of never having to
reconcile a half-written file. (Writes go through a temp file plus `os.replace`, so a killed process
leaves either the previous complete file or no file.)

### With the default LOCAL judge, a full re-score is FREE

The default judge is the same `roles.DEFAULT_JUDGE_MODEL` (`google/gemma-4-E4B-it`) behind vLLM
that plays the oracle and the patient, so the entire grid costs GPU-hours and **$0 in API**. That is the point of Exp4, and it is
the thing that was impossible in Exp3: there the lake was several hundred dollars of irreplaceable
vendor calls, and "delete the partition and re-run it" was not a move anyone could make. Here it
is -- drop `judge=<tag>/rep=<r>/metric=<M>/` and re-run this notebook.

The freedom ends the moment `JUDGE_PROVIDER` names a vendor API. Then the lake is expensive again
and Exp3's rules apply.

### Where this runs: on the GPU that serves the judge

Colab (the A100 80 GB the arms train on) or a GPU server over SSH -- **not** the local 12 GB
card. The E4B judge is 14.89 GiB of weights before any KV cache; at `0.85 x 12 GB = 10.2 GiB`
vLLM cannot even load them, and the E2B fallback would leave under 1 GiB of KV at
`max_model_len 16384`. Locally, this notebook is for smoke tests against a fake or remote
endpoint only. The setup cell therefore carries the same Drive-mount preamble as the trainer
notebooks; `eda/` is pushed to Drive next to `code/` (additively -- never `data/`).

### This notebook is NOT a family and writes nothing under `results/`

There is no `EdaConfig` and no `notebook_setup` here, and `tools/render_results.py` never executes
it (`notebooks/scoring/` is exempt from the `FAMILIES` map on purpose). Its only output is the score
lake under `data/`. **Do not look for its figures or tables -- it has none.**

Cell order below is deliberate: **setup -> configure -> serve -> sanity gate -> plan -> run ->
verify -> prompt-length gate**. The sanity gate sits before the plan because a degenerate grader
is invisible after the fact; the prompt-length gate sits after the run because it measures the
real Exp4 conversations that were just graded against the server's `max_model_len` -- the
Phase 2 measurement the spec asks for (CLAUDE.md § VRAM budget).

## 0. Colab: the pinned stack

**Byte-identical to cell 0 of both trainer notebooks, on purpose.** Scoring with the
local Gemma judge starts a vLLM server through the same `tools/vllm_serve.py` the
trainers use, so it needs the same stack -- and Colab gives every notebook its own VM,
so running the trainer's install cell does not install anything for this one.

The cell refuses to install outside Colab, so opening this notebook locally is safe:
it prints what is missing and changes nothing. A **vendor judge needs none of it** --
`gpt-4o-mini` and `claude-haiku-4-5` are HTTP calls, no GPU, no vLLM -- so for those two
this notebook runs locally from the repo `.venv` and this cell is a no-op.

On a fresh Colab runtime it installs once and then **raises on purpose** to stop
Run-all: restart the session, re-run this cell (one line, skips) and continue.
`_selfcheck`'s `install cell parity` check keeps the three copies identical.


In [ ]:
# =============================================================================
#  CELL 0 -- environment. This cell RUNS ITSELF; there is nothing to uncomment.
# =============================================================================
# GUARDED, not commented out. Three things it refuses to do:
#
#   * reinstall on a runtime that is already correct -- minutes of churn, and
#     re-installing vLLM re-resolves torch underneath a stack that was fine;
#   * install anything OUTSIDE Colab -- this notebook is importable locally for
#     smoke tests, and a vLLM/CUDA install into the repo's .venv is not
#     something to do by accident;
#   * install in the wrong order -- vLLM brings its own torch wheel, so it goes
#     FIRST and the pinned stack is layered on top. Installing vLLM last
#     silently replaces the torch the training stack was resolved against.
#
# So: a fresh runtime installs once and asks for a restart; every later run
# prints one line and moves on.
# ⚠ Byte-identical to the GRPO notebook's cell 0 on purpose -- the two install
# cells drifting apart is exactly how one method ends up on a different stack.
AUTO_INSTALL = True        # False -> report what is missing, change nothing

import os
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

IS_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))

# The repo's validated set (requirements.txt). torch is pinned SEPARATELY below,
# by CUDA build and not just by version -- see TORCH_TRIO.
PINNED = {
    "accelerate": "1.13.0",
    "datasets": "4.8.5",
    "huggingface_hub": "1.14.0",
    "numpy": "2.4.4",
    "openai": "2.36.0",
    "pandas": "3.0.3",
    "peft": "0.19.1",
    "safetensors": "0.7.0",
    "tensorboard": "2.20.0",
    "transformers": "5.8.1",
    "trl": "1.4.0",
}
# vLLM is PINNED. An unpinned `pip install -U vllm` resolves to whatever PyPI has
# today, and on 2026-09-14 that was 0.29.0 (released 2026-09-09), which requires
# transformers>=5.10.4 and torch==2.13.0 -- re-asserting transformers==5.8.1 on
# top of it then fails Gate 1 below, on the first Colab session, before anything
# runs. 0.26.0 (2026-07-25) pins torch==2.11.0 (the torch the pinned stack was
# validated on locally) and every one of its requirements (transformers>=5.5.3,
# openai>=2.0.0, safetensors>=0.6.2, torchvision==0.26.0, torchaudio==2.11.0) is
# satisfied by PINNED. Bumping it is a STACK change: re-run `smoke.py roles`, and
# re-verify the trl line references in CLAUDE.md § Hyperparameters first. Never
# set it back to None ("latest that clears VLLM_MIN"). Gemma 4 needs at least
# VLLM_MIN, per the official vLLM Gemma 4 recipe; an older build is treated as
# work to do, not as a warning to act on by hand: it cannot serve the grader at
# all, so section 3 would fail either way.
PINNED_VLLM = "0.26.0"      # torch 2.11.0 -- read the note above before changing
VLLM_MIN = (0, 19, 1)
# ... and the CUDA BUILD is pinned too, because a version pin alone is not enough.
# Measured on the first Colab session (2026-09-14): the PyPI vllm==0.26.0 wheel is
# a CUDA 13 build (its _C extension needs libcudart.so.13), while Colab's
# pre-installed torch is a CUDA 12 build whose version ALREADY satisfies
# `torch==2.11.0`, so pip left it in place and `import vllm` died on the missing
# libcudart.so.13. So: the +cu129 vLLM wheel from the GitHub release, and the
# torch trio FORCED to the matching +cu129 build from the PyTorch index (a
# `-U` would not do -- pip ranks a `+cu130` local tag above `+cu129`). CUDA 12.9
# runs on any driver >= 525, i.e. every Colab card. Both sides agree on CUDA 12.9.
# The trio's versions are vLLM 0.26.0's own pins (torch==2.11.0, torchvision==0.26.0,
# torchaudio==2.11.0); bump them together with PINNED_VLLM, never alone.
VLLM_CUDA = "cu129"
VLLM_WHEEL_URL = (f"https://github.com/vllm-project/vllm/releases/download/v{PINNED_VLLM}/"
                  f"vllm-{PINNED_VLLM}+{VLLM_CUDA}-cp38-abi3-manylinux_2_28_x86_64.whl")
TORCH_INDEX = f"https://download.pytorch.org/whl/{VLLM_CUDA}"
TORCH_TRIO = {"torch": "2.11.0", "torchvision": "0.26.0", "torchaudio": "2.11.0"}
VLLM_WANT = f"{PINNED_VLLM}+{VLLM_CUDA}"          # what importlib.metadata reports for the wheel


def _v(pkg):
    """Installed version string, or None. Reads package METADATA -- imports nothing.

    That matters here: this cell must not pull torch in, both for the sm_120
    import-order contract and because section 3 has to start vLLM first.
    """
    try:
        return version(pkg)
    except PackageNotFoundError:
        return None


def _older_than(ver, minimum):
    """True iff *ver* parses as a release older than *minimum*. Unparseable -> False."""
    try:
        return tuple(int(x) for x in ver.split("+")[0].split(".")[:3]) < minimum
    except (AttributeError, ValueError):
        return False           # a dev/nightly build -- do not cry wolf


_off_pin = {p: (_v(p), want) for p, want in PINNED.items() if _v(p) != want}
_vllm = _v("vllm")
# The build tag is part of the pin: a bare "0.26.0" here is the PyPI CUDA 13 wheel.
_vllm_bad = (_vllm is None or _older_than(_vllm, VLLM_MIN) or _vllm != VLLM_WANT)
_torch_off = {p: (_v(p), f"{want}+{VLLM_CUDA}") for p, want in TORCH_TRIO.items()
              if _v(p) != f"{want}+{VLLM_CUDA}"}
# Colab pre-bakes torchao < 0.16.0, and peft 0.19.1 does not merely ignore it:
# get_peft_model's LoRA dispatcher calls dispatch_torchao, which RAISES against
# that version, so attaching the adapter fails outright. Nothing here uses it.
_torchao = _v("torchao")
_work = bool(_off_pin) or _vllm_bad or bool(_torch_off) or _torchao is not None


def _report():
    for _p, (_have, _want) in sorted(_off_pin.items()):
        print(f"  {_p}: have {_have}, want {_want}")
    if _vllm is None:
        print("  vllm: not installed")
    elif _older_than(_vllm, VLLM_MIN):
        print(f"  vllm {_vllm}: older than {'.'.join(map(str, VLLM_MIN))}, the Gemma 4 floor")
    elif _vllm != VLLM_WANT:
        print(f"  vllm: have {_vllm}, want {VLLM_WANT} (PINNED_VLLM + VLLM_CUDA)")
    for _p, (_have, _want) in sorted(_torch_off.items()):
        print(f"  {_p}: have {_have}, want {_want} (the {VLLM_CUDA} build from {TORCH_INDEX})")
    if _torchao is not None:
        print(f"  torchao {_torchao}: installed, and peft 0.19.1 raises against it")


# The packages THIS cell owns. `pip check` (Gate 1) fails only on a conflict whose
# REQUIRING package is one of these: stock Colab already carries conflicts that are
# none of our business (google-colab's own pandas/numpy pins, say), and a gate that
# raised on any line would refuse a runtime that is fine for us. A conflict raised by
# vllm / torch / the pinned stack (vLLM wanting a newer transformers than the pin, a
# torch moved under trl) is a stack that would fail later, somewhere less legible.
_OWNED = ({p.lower().replace("_", "-") for p in PINNED} | {"vllm"}
          | {p.lower() for p in TORCH_TRIO})


def _pip_check_gate():
    """Gate 1: the resolved set is consistent for every package this stack owns."""
    print("  $ pip check")
    _res = subprocess.run([sys.executable, "-m", "pip", "check"],
                          capture_output=True, text=True)
    _lines = [ln.strip() for ln in _res.stdout.splitlines() if ln.strip()]
    if _res.returncode == 0 or not _lines:
        print("  pip check: no broken requirements"
              + (f" (exit {_res.returncode}: {_res.stderr.strip()[-300:]})"
                 if _res.returncode else ""))
        return
    _owned = [ln for ln in _lines if ln.split()[0].lower().replace("_", "-") in _OWNED]
    for ln in _lines:
        if ln not in _owned:
            print(f"  pip check (outside this stack, tolerated): {ln}")
    if _owned:
        raise RuntimeError("pip check: the training/serving stack is inconsistent --\n  "
                           + "\n  ".join(_owned))
    print(f"  pip check: {len(_lines)} pre-existing conflict(s) outside this stack, none in it")


if not _work:
    print(f"environment OK -- {len(PINNED)} pinned packages match, vllm {_vllm}, "
          f"torch {_v('torch')}, torchao absent")
    if IS_COLAB:
        _pip_check_gate()      # a warm runtime is re-checked too: the gate is cheap
elif not AUTO_INSTALL:
    print("AUTO_INSTALL=False -- nothing was changed. Outstanding:")
    _report()
elif not IS_COLAB:
    print("NOT Colab -- refusing to install (this would write into the local .venv).")
    print("Outstanding, for reference:")
    _report()
else:
    def _pip(*args):
        print(f"  $ pip {' '.join(args)}")
        subprocess.check_call([sys.executable, "-m", "pip", *args])

    if _torch_off or _vllm_bad:
        # torch FIRST, and by exact build: --force-reinstall --no-deps swaps the three wheels
        # for the cu129 ones whatever was there (Colab's own build satisfies the bare version,
        # and -U would keep a "+cu130"), then a plain install fills in their cu12 runtime libs
        # (nvidia-*-cu12 at the 12.9 pins). Only then vLLM, whose torch==2.11.0 pin is now met
        # by the cu129 build so it fetches no torch of its own.
        _exact = [f"{p}=={v}+{VLLM_CUDA}" for p, v in TORCH_TRIO.items()]
        print(f"pinning the torch trio to the {VLLM_CUDA} build ({TORCH_INDEX}) -- a few minutes")
        _pip("install", "-q", "--force-reinstall", "--no-deps",
             "--extra-index-url", TORCH_INDEX, *_exact)
        _pip("install", "-q", "--extra-index-url", TORCH_INDEX, *_exact)
    if _vllm_bad:
        print(f"{'installing' if _vllm is None else f'replacing vLLM {_vllm} ->'} vLLM "
              f"{VLLM_WANT} from the GitHub release -- takes several minutes")
        _pip("install", "-q", "--extra-index-url", TORCH_INDEX, VLLM_WHEEL_URL)
    if _off_pin or _vllm_bad or _torch_off:
        # The FULL list, not just the off-pin subset: a fresh vLLM may have moved
        # numpy/transformers underneath it, so re-assert the whole validated set.
        print("pinning the training stack on top of vLLM's torch")
        _pip("install", "-q", *[f"{p}=={v}" for p, v in PINNED.items()])
    if _torchao is not None:
        print(f"removing torchao {_torchao} (peft 0.19.1 raises inside its LoRA dispatcher)")
        _pip("uninstall", "-y", "-q", "torchao")

    # Gate 1: the resolved set is CONSISTENT for the packages this stack owns (see
    # _pip_check_gate above for why foreign conflicts are printed, not raised on).
    _pip_check_gate()

    # Gate 2: vLLM IMPORTS -- probed in a SUBPROCESS. Never import vllm (or torch) in THIS
    # kernel: section 3's server must claim its pre-allocation before torch initialises CUDA
    # here, and section 5's trl-before-torch order must survive on the local sm_120 card.
    _probe = subprocess.run(
        [sys.executable, "-c",
         "import torch, vllm; print(vllm.__version__, '| torch', torch.__version__, "
         "'cuda', torch.version.cuda, '| cuda available', torch.cuda.is_available())"],
        capture_output=True, text=True)
    if _probe.returncode != 0:
        raise RuntimeError("vLLM is installed but does not import in a fresh interpreter:\n"
                           + _probe.stderr[-2000:])
    print(f"  vllm imports in a fresh interpreter: {_probe.stdout.strip()}")

    print("\n" + "=" * 74)
    print("  RESTART THE RUNTIME NOW  (Runtime -> Restart session), then run the mount")
    print("  cell and this cell again -- it will print one line and skip. Installing")
    print("  over modules the kernel has already imported leaves it holding the old ones.")
    print("=" * 74)
    # Raised on purpose so Run-all STOPS here: every cell below would otherwise run on the
    # stale, half-imported stack and fail somewhere that looks like a training bug.
    raise RuntimeError("Runtime restart required: Runtime > Restart session, then re-run the "
                       "mount cell and this cell")

# bitsandbytes is only needed for USE_4BIT=True, which Exp4 never runs (4-bit induced ~30x
# more phrase-loop degeneration on this same base model in Exp2, which moved the whole score
# axis). If you ever flip that toggle: pip install bitsandbytes==0.49.2

## 1. Setup

`from eda_analysis import scoring` is written out explicitly, and that is the whole design:
`scoring` is deliberately absent from the package's lazy attribute map, so no analysis import can
reach the one module that talks to a grader by accident. **This import is the moment someone chooses
to spend** (GPU-hours on the open stack, real money on a vendor judge).

Importing `eda_analysis` also prepends `Exp4_OpenStack/code` to `sys.path`, which is why `roles`,
`tools.vllm_serve` and `tools.oracle_sanity` resolve below to the single canonical copies the
trainers themselves run.

On Colab the kernel starts in `/content`, where a walk-up finds no `eda_analysis/`; the cell
therefore mounts Drive and `chdir`s into this notebook's own folder first, exactly as the
trainer notebooks do. After a runtime restart, **re-run this cell** before anything else.

In [ ]:
import os
import sys

# Scoring runs where the judge is SERVED: Colab (A100 80 GB) or a GPU server over SSH. The local
# 12 GB card cannot hold the E4B judge (14.89 GiB of weights alone; at 0.85 x 12 GB = 10.2 GiB the
# server cannot even load them), so locally this notebook is for smoke tests against a fake or
# remote endpoint only. Hence the same Drive preamble the trainer notebooks carry: on Colab the
# kernel starts in /content, where the walk-up below finds no eda_analysis/ and would die on a
# bare ImportError. Re-run this cell after every runtime restart.
COLAB_EDA_DIR = "/content/drive/MyDrive/Thesis_PTO_GRPO/Exp4_OpenStack/eda/notebooks/scoring"

if "google.colab" in sys.modules or os.environ.get("COLAB_RELEASE_TAG"):
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")
    if os.path.isdir(COLAB_EDA_DIR):
        os.chdir(COLAB_EDA_DIR)
    else:
        print(f"WARNING: COLAB_EDA_DIR does not exist: {COLAB_EDA_DIR} -- push eda/ to Drive "
              f"(additively, next to code/; never data/) and re-run this cell.")

# Find eda/ -- the directory holding eda_analysis/ -- from wherever this notebook was opened.
_eda_dir = os.path.abspath(os.getcwd())
while _eda_dir != os.path.dirname(_eda_dir) and not os.path.isdir(
        os.path.join(_eda_dir, "eda_analysis")):
    _eda_dir = os.path.dirname(_eda_dir)
if not os.path.isdir(os.path.join(_eda_dir, "eda_analysis")):
    raise RuntimeError(
        f"no eda_analysis/ package above {os.getcwd()}. Open this notebook from "
        f"Exp4_OpenStack/eda/notebooks/scoring/ (on Colab: point COLAB_EDA_DIR at where eda/ "
        f"was pushed on Drive and re-run this cell)."
    )
if _eda_dir not in sys.path:
    sys.path.insert(0, _eda_dir)

import pandas as pd
from IPython.display import display

# EXPLICIT on purpose -- see the note above. `data` is the read side; `scoring` is the write side.
from eda_analysis import data, scoring
from eda_analysis.constants import DATA_DIR, EVAL_SCORES_DIR, N_PERSONAS, available_judge_tags

# Canonical trainer-side modules, reachable because importing eda_analysis put code/ on sys.path.
from core.runtime import authenticate
from roles import DEFAULT_JUDGE_MODEL, make_binding, reset_client_cache
from tools import oracle_sanity
from tools.vllm_serve import ensure_alive, report_weights_gib, serve_roles

print("eda root   :", _eda_dir)
print("data root  :", DATA_DIR)
print("score lake :", EVAL_SCORES_DIR)
print("judges already in the lake:", ", ".join(available_judge_tags()) or "(none yet)")
print("arms with conversations on disk:", len(data.discover_arms()))


## 2. Configuration

Every knob this notebook has. Nothing else configures a scoring pass.

The judge is the one setting that must never be implicit: `JUDGE_MODEL` decides the `judge=<tag>`
directory every score is filed under, and the tag is derived *inside* `score_model_state` from the
binding that actually does the grading -- so the folder cannot disagree with the grader that filled
it. `run_scoring` additionally refuses a plan whose `judge` column does not match the binding it was
handed.

`REP = 0` is the full-grid draw every family reads. Use `rep >= 1` only for repeatability re-draws;
a mismatch would overwrite the draw everything else reports.

`METRICS = None` means all eight stored instruments (`Q1`, `Q2`, `WAI_SR`, `CSQ8`, `MI_SAT`, `MITI`,
`PCT`, `MICI`). `Q1Q2` is a **composite** -- the loader averages Q1 and Q2 after reading, exactly as
`core.oracle.score_conversation` does -- so it is never scored and never stored; asking for it
raises.

In [ ]:
# ---------------------------------------------------------------------------
# JUDGE -- who grades. Decides the judge=<tag> partition; never leave it implicit.
# ---------------------------------------------------------------------------
# Exp4 grades the SAME conversations with several judges, into SEPARATE partitions that every
# family then puts side by side. Pick ONE preset per run. A second run under another preset
# writes a second partition and re-scores nothing, so the three passes are independent and each
# one resumes on its own.
#
#   preset     model                  provider        cost   GPU   held out?
#   --------   --------------------   -------------   -----  ---   ------------------------------
#   "gemma"    google/gemma-4-E4B-it  openai_compat   $0     YES   NO -- the same model as the
#                                                                  TRAINING oracle, so the primary
#                                                                  number is train-on-test, just
#                                                                  as in Exp3. ~14.6 GiB of
#                                                                  weights: Colab or another 80 GB
#                                                                  card, never the 12 GB local one.
#   "gpt"      gpt-4o-mini            openai          BILLS  no    YES -- it trained nothing here,
#                                                                  and Exp3 holds thousands of its
#                                                                  scores to calibrate against.
#   "claude"   claude-haiku-4-5       anthropic       BILLS  no    YES -- and it is Exp3's own
#                                                                  second judge, so the two
#                                                                  experiments share a yardstick.
#
# Both vendor judges are pure HTTP: no vLLM, no GPU, nothing to serve. Run THOSE locally from the
# repo .venv (cell 0 is a no-op off Colab) and keep the GPU session for the Gemma pass.
#
# WARNING: never average raw scores across judges. The level offset between them is real and
# model-dependent -- the Gemma-vs-gpt-4o-mini sanity run measured -0.93 on Q1 -- so an average
# applies a silent, model-dependent shrinkage to every effect. Combine CONTRASTS, not levels.
JUDGE_PRESET = "gemma"                 # "gemma" | "gpt" | "claude" | "" = set the fields by hand

_JUDGE_PRESETS = {                     # preset -> (model, provider, serve a local server?)
    "gemma":  (DEFAULT_JUDGE_MODEL, "openai_compat", True),
    "gpt":    ("gpt-4o-mini",       "openai",        False),
    "claude": ("claude-haiku-4-5",  "anthropic",     False),
}
if JUDGE_PRESET:
    JUDGE_MODEL, JUDGE_PROVIDER, _SERVE_LOCAL_DEFAULT = _JUDGE_PRESETS[JUDGE_PRESET]
else:                                  # hand-set: spell all three out, no preset magic
    JUDGE_MODEL          = DEFAULT_JUDGE_MODEL
    JUDGE_PROVIDER       = "openai_compat"
    _SERVE_LOCAL_DEFAULT = True
JUDGE_BASE_URL = ""                    # "" -> filled in by the serve cell below

# ---------------------------------------------------------------------------
# LOCAL SERVER (openai_compat only; ignored for a vendor judge)
# ---------------------------------------------------------------------------
SERVE_LOCAL = _SERVE_LOCAL_DEFAULT     # False = a server someone else started (set JUDGE_BASE_URL);
                                       # always False for a vendor judge -- there is nothing to serve
SERVE_PORT = 8000
# vLLM PRE-ALLOCATES this fraction of the card and never gives it back. On the target A100 80 GB
# with an otherwise IDLE GPU: 0.85 x 80 = 68 GiB = 14.89 GiB of E4B weights + a ~50 GiB KV pool --
# fine, and far more concurrency than CONCURRENCY ever asks for. If a TRAINER shares the card,
# drop this to the trainers' cell-1 value, 0.50 = roles.default_serve_util(E4B), and start
# the server FIRST: training memory is the spiky side. Applies only when the cell below actually
# LAUNCHES a server -- an adopted one keeps its own. (Local 12 GB card: 0.85 x 12 = 10.2 GiB is
# LESS than the E4B weights -- it cannot serve this judge; see the setup cell.)
SERVE_GPU_MEMORY_UTILIZATION = 0.85
# Do NOT lower this. Measured on 192 real Exp3 transcripts, full Q2 oracle prompts reach 10,042
# tokens; at 8192 about 2.1% of Q2 and 1.0% of Q1 prompts would not fit -- and those are the
# LONGEST conversations, whose length varies by arm and by K. The dropout would be arm-dependent
# bias on the headline metric, and it would be silent, because an unscoreable conversation is
# simply absent rather than an error. Give memory back via SERVE_GPU_MEMORY_UTILIZATION instead.
SERVE_MAX_MODEL_LEN = 16384

# ---------------------------------------------------------------------------
# WHAT TO SCORE
# ---------------------------------------------------------------------------
REP = 0                                # 0 = the full-grid draw every family reads
METRICS = None                         # None = all eight stored instruments; or e.g. ["Q1", "Q2"]

ARM_METHODS = None                     # e.g. ["PTO"]      -- None = no filter
ARM_KS = None                          # e.g. [0]
ARM_MODES = None                       # e.g. ["greedy"]; "" selects GRPO arms
ARM_LABELS = None                      # e.g. ["GRPO_LA5"] -- a DISPLAY key, can match >1 arm

# ---------------------------------------------------------------------------
# HOW HARD TO TRY (per grading call)
# ---------------------------------------------------------------------------
CONCURRENCY       = scoring.DEFAULT_CONCURRENCY      # in-flight calls WITHIN one model state
MAX_TOKENS        = scoring.JUDGE_MAX_TOKENS         # clipped JSON -> retries -> holes; raise first
MAX_RETRIES       = scoring.JUDGE_MAX_RETRIES        # attempts per call, first one included
REQUEST_TIMEOUT   = scoring.JUDGE_REQUEST_TIMEOUT    # PER ATTEMPT, seconds
MIN_SUCCESS_RATIO = scoring.MIN_SUCCESS_RATIO        # below this a partition is NOT written at all

# ---------------------------------------------------------------------------
# THE SANITY GATE (cell 4)
# ---------------------------------------------------------------------------
SANITY_QUICK = False                   # False = all 12 fixture transcripts; True = the 2 extremes
SANITY_QUESTIONNAIRES = (1, 2)         # the fixture carries reference scores for Q1/Q2 only
# Match the scoring pass: a gate run at a gentler concurrency can pass against a server that
# falls over under the real load.
SANITY_CONCURRENCY = CONCURRENCY
SANITY_REPORT_PATH = ""                # "" = do not archive; a dir gets oracle_sanity.json

# ---------------------------------------------------------------------------
# THE PROMPT-LENGTH GATE (cell 8) -- the Phase 2 measurement the spec asks for
# ---------------------------------------------------------------------------
PROMPT_LENGTH_GATE = True              # False = skip the measurement (say so in the run notes)
PROMPT_LENGTH_QUESTIONNAIRES = (1, 2)  # Q2 carries the longest rubric; None = every stored
                                       # instrument (widen once prompt_length_report handles the
                                       # nested MITI/PCT/MICI prompts)

# ---------------------------------------------------------------------------
# RUN
# ---------------------------------------------------------------------------
DRY_RUN = False                        # True = validate the plan, make no calls

print(f"judge      : {JUDGE_PROVIDER}:{JUDGE_MODEL}")
print(f"rep        : {REP}")
print(f"metrics    : {'all eight stored' if METRICS is None else ', '.join(METRICS)}")
print(f"call policy: concurrency={CONCURRENCY} max_tokens={MAX_TOKENS} "
      f"max_retries={MAX_RETRIES} timeout={REQUEST_TIMEOUT:.0f}s")
if JUDGE_PROVIDER != "openai_compat":
    print("\n*** WARNING: this is a VENDOR judge. Every call below is BILLED, and a re-score is "
          "no longer free. Check the plan cell's cost line before running. ***")
if JUDGE_PROVIDER == "anthropic":
    print("    Claude path: the rubric schemas lose their numeric and array-length "
          "constraints\n    (its Messages API rejects them) and carry those constraints as "
          "description text\n    instead. The serve cell runs check_rubric_parity() first: "
          "that gate is FREE, and it\n    is what stands between a stripped constraint and "
          "silent biased missingness.")


## 3. Serve the judge

`serve_roles` is **idempotent**: a healthy server already listening on the port and serving the
right model is *adopted*, not duplicated, so re-running this cell is safe. A server answering on
that port with a **different** model is a hard error rather than an adoption -- silently grading the
whole lake with the wrong model is the most expensive failure available here.

Nothing is started for a vendor judge, and nothing is started when `SERVE_LOCAL = False` (then
`JUDGE_BASE_URL` must name the endpoint).

> An `openai_compat` binding with **no** `base_url` does not fail -- the OpenAI SDK quietly points
> at `api.openai.com`, billing a run that exists to cost $0 while the lake records the *local*
> model's tag. `scoring.judge_binding_for` warns about it and `score_model_state` refuses outright;
> this cell fills the URL in from the server it just brought up, so the case should never arise.

In [ ]:
SERVER_HANDLES = {}
_base_url = (JUDGE_BASE_URL or "").strip() or None

if JUDGE_PROVIDER != "openai_compat":
    print(f"[Run_Eval] {JUDGE_PROVIDER} judge -- nothing to serve.")
elif not SERVE_LOCAL:
    if not _base_url:
        raise ValueError(
            "SERVE_LOCAL is False, so JUDGE_BASE_URL must name the running server. An "
            "'openai_compat' binding with no base_url sends every call to api.openai.com -- a "
            "vendor bill on a $0 experiment, filed under the local model's judge tag."
        )
    print(f"[Run_Eval] using an externally managed server at {_base_url} (starting nothing)")
else:
    _wired, SERVER_HANDLES = serve_roles(
        {"judge": make_binding("openai_compat", JUDGE_MODEL, base_url=_base_url)},
        base_port=SERVE_PORT,
        gpu_memory_utilization=SERVE_GPU_MEMORY_UTILIZATION,
        max_model_len=SERVE_MAX_MODEL_LEN,
    )
    _base_url = _wired["judge"].base_url
    for _handle in SERVER_HANDLES.values():
        _weights = report_weights_gib(_handle)
        print(f"[Run_Eval] {_handle.model}: weights "
              f"{'unknown (startup log not parsed)' if _weights is None else f'{_weights:.2f} GiB'}"
              f", max_model_len={_handle.spec.max_model_len}")

# A VENDOR judge needs an API key, and nothing else in eda/ resolves one: roles.make_client
# reads os.environ only, while the trainers route through core.runtime.get_secret (env ->
# Colab secret -> openai_key.txt in the workspace root). Without this the identical machine
# that trains fine fails mid-sweep with "No API key for role binding", naming none of the
# three places the key actually lives. Local judges need nothing, so nothing is asked for.
if JUDGE_PROVIDER == "openai":
    authenticate(hf=False, openai=True)
elif JUDGE_PROVIDER == "anthropic":
    authenticate(hf=False, anthropic=True)

# FREE pre-flight, and the reason a Claude judge is allowed here at all. Anthropic's Messages API
# rejects minimum/maximum/minItems/maxItems, so every rubric schema goes through
# core.oracle.strip_unsupported_constraints, which folds each constraint into `description` text
# rather than deleting it. For the five flat rubrics minItems == maxItems == n_questions is the
# ONLY guarantee of one score per item: drop it silently and the hardest conversations come back
# wrong-length, fail the validation ladder, and land as NaN rows -- biased missingness on the
# headline metric, invisible in the output. This asserts every stripped constraint was restated
# and that nothing else about the schema moved. It costs nothing, so it runs before the key is
# ever used on a real call.
if JUDGE_PROVIDER == "anthropic":
    _parity = scoring.check_rubric_parity()
    if not _parity["ok"]:
        raise RuntimeError(
            "RUBRIC PARITY FAILED -- refusing to spend on a Claude judge whose schema lost a "
            f"constraint without restating it: {_parity['problems'][:5]}"
        )
    print(f"[Run_Eval] rubric parity OK: {len(_parity['checked'])} rubrics, "
          f"{_parity['n_stripped']} constraints stripped and all restated in description text")

JUDGE = scoring.judge_binding_for(
    JUDGE_MODEL,
    provider=JUDGE_PROVIDER,
    base_url=_base_url,
    request_timeout=REQUEST_TIMEOUT,
    max_retries=MAX_RETRIES,
)
JUDGE_TAG = scoring.judge_tag(JUDGE)

print(f"[Run_Eval] judge     : {JUDGE.provider}:{JUDGE.model}")
print(f"[Run_Eval] endpoint  : {JUDGE.base_url or '(provider default)'}")
print(f"[Run_Eval] writes to : data/eval_scores/judge={JUDGE_TAG}/rep={REP}/metric=<M>/...")

## 4. SANITY GATE -- stop here if the grader is not a measuring instrument

**Do not skip this cell.** An open-weights grader fails in two ways and only one of them is loud:

1. **LOUD** -- it ignores the schema or returns the wrong number of item scores. `core.oracle`'s
   validation ladder catches it, but "caught" is not "visible": what a run sees is a rising retry
   count and then *biased missingness*, because the conversations the grader found hardest are
   exactly the ones that disappear. Hence the gate demands `schema_valid_rate == 1.000`, not 0.98.
2. **SILENT, AND WORSE** -- it honours the schema perfectly and returns degenerate scores: every
   item a 4, near-zero variance across conversations a validated grader placed four points apart.
   That parses, passes every rung of the ladder, writes valid parquet, and yields a grader that
   cannot tell any two arms apart. Nothing downstream flags it. The contrast tables come back at
   ~0, the plots look flat, and it reads like a finding.

A four-arm grid of ten model states each is `4 x 10 x 8 x 96 = 30,720` graded cells. Producing all
of them with a degenerate grader and discovering it in the contrast tables is the expensive path;
this gate is 12 transcripts x 2 rubrics = 24 calls.

**Hard gates (block the run):** perfect schema compliance on every requested rubric, and
per-conversation SD at or above `MIN_SCORE_SD` on every rubric *and* on the pooled reward.
**Soft (reported, never blocking):** Spearman rank agreement with the frozen `gpt-4o-mini`
reference, and the level offset. Exp3 and Exp4 are **not on the same score axis** -- a different
grader sits systematically higher or lower, so an offset of a point or more is expected and says
nothing about fitness. What survives the axis change is the ORDERING and the SPREAD.

The gate calls `core.oracle.get_evaluation_json` -- the same prompt builder, schema shim, validation
ladder and retry policy this notebook's scoring path uses. A green report against a private scorer
would prove nothing about the run.

Widening `SANITY_QUESTIONNAIRES` beyond `(1, 2)` is legitimate: the hard gates need no reference at
all. The soft columns just come back `n/a` for the six rubrics the fixture has no reference for --
which means "no reference exists", never "the grader failed".

In [ ]:
sanity_report = scoring.run_async(oracle_sanity.run_sanity(
    JUDGE,
    questionnaire_ids=SANITY_QUESTIONNAIRES,
    quick=SANITY_QUICK,
    concurrency=SANITY_CONCURRENCY,
    max_tokens=MAX_TOKENS,          # same budget the scoring pass uses -- a gate at a LARGER
    max_retries=MAX_RETRIES,        # budget tests a configuration this notebook will not run
    request_timeout=REQUEST_TIMEOUT,
    progress=True,
))

print(oracle_sanity.format_report(sanity_report))

if SANITY_REPORT_PATH:
    print("report written to", oracle_sanity.write_report(sanity_report, SANITY_REPORT_PATH))

_passed, _reasons = oracle_sanity.check_gates(sanity_report)
if not _passed:
    raise RuntimeError(
        "ORACLE SANITY FAILED -- refusing to score with this grader:\n  "
        + "\n  ".join(_reasons)
        + "\n\nScoring the grid anyway would write tens of thousands of valid-looking cells that "
          "mean nothing, and nothing downstream would flag it. Check, in order: thinking mode is "
          "off for this binding, MAX_TOKENS is not clipping the JSON, and the pinned vLLM accepts "
          "strict json_schema (core.oracle.set_openai_compat_strict(False) if it 400s on the key). "
          "Then re-run this cell."
    )

print(f"\nSANITY GATE PASSED -- {JUDGE.provider}:{JUDGE.model} honours the schema and separates "
      f"the fixture. Proceed.")

## 5. Plan -- what is missing, and what it will cost

Nothing is scored here. `discover_scorable` lists one row per `(arm, model state, metric)` still
missing from the lake, and `estimate_calls` says what running it implies -- `$0 (local)` for the
open stack, a labelled projection for a vendor judge, and `unknown` (never `$0`) for a vendor judge
with no pricing row.

An **empty plan is the normal, correct result of a second run.** It is not a failure.

Arms are discovered from `data/conversations/` -- a run becomes scoreable the moment its
conversations land, and there is no registry to edit. One call per `(conversation, rubric)`: each
parquet is one rubric and each rubric is one schema-constrained request.

> A state whose conversation count later GROWS -- a repair pass that regenerates two failed personas
> -- re-enters the plan and is re-scored in full, all 96 calls, not two. Free on the local stack;
> check the cost line first on a vendor judge.

In [ ]:
ARMS = data.filter_arms(
    data.discover_arms(),
    methods=ARM_METHODS, ks=ARM_KS, modes=ARM_MODES, arm_labels=ARM_LABELS,
)
METRIC_KEYS = list(scoring.STORED_METRICS) if METRICS is None else list(METRICS)

plan = scoring.discover_scorable(ARMS, judge=JUDGE_TAG, rep=REP, metrics=METRIC_KEYS)
estimate = scoring.estimate_calls(plan, binding=JUDGE, max_retries=MAX_RETRIES)

print(f"arms            : {len(ARMS)}"
      f"  [{', '.join(a.label for a in ARMS) if ARMS else 'none on disk'}]")
print(f"model states    : {sum(len(a.iters) for a in ARMS)}")
print(f"metrics         : {', '.join(METRIC_KEYS)}")
print(f"partitions to do: {estimate['n_partitions']}")
print(f"grading calls   : {estimate['n_calls']:,}"
      f"   (worst case with retries: {estimate['n_calls_worst_case']:,})")
print(f"cost            : {estimate['cost']}")
for _note in estimate["notes"]:
    print(f"  note: {_note}")

if plan.empty:
    print("\nNothing to score: every requested partition already exists for "
          f"judge={JUDGE_TAG}, rep={REP}. Skip to the verify cell.")
else:
    _by_arm = (plan.groupby(["arm_label", "experiment_name"], as_index=False)
                   .agg(states=("state_index", "nunique"),
                        partitions=("metric", "size"),
                        calls=("n_conversations", "sum")))
    print(f"\nMissing partitions by arm (judge={JUDGE_TAG}, rep={REP}):")
    display(_by_arm)

    _by_metric = (pd.Series(estimate["by_metric"], name="calls")
                    .rename_axis("metric").reset_index()
                    .sort_values("metric", kind="mergesort"))
    print("Grading calls by metric:")
    display(_by_metric)

## 6. Run

Model states run **sequentially**; each already fans out to `CONCURRENCY` in-flight calls, so
running two at once would double the load on the server without touching the bound that was
actually configured. Sequential also makes the resume granularity exactly one file.

A state that fails is recorded with `status="error"` and **the run continues** -- one state that
could not be graded should not cost the twenty that could, and the failure comes back in the frame
rather than vanishing. Those partitions stay in the plan, so re-running the previous two cells
retries exactly them.

A partition where fewer than `MIN_SUCCESS_RATIO` of conversations graded successfully is **not
written at all**. That is deliberate: a written file counts as done forever, so a mostly-empty
partition would freeze a broken grader's output into the lake and every downstream table would
silently be computed over the conversations that happened to be easy to score.

> Interrupting this cell is safe. Completed partitions are on disk; re-run the plan cell (the `plan`
> frame is now stale) and then this one.

In [ ]:
for _name, _handle in SERVER_HANDLES.items():
    _restarts_before = _handle.restarts
    ensure_alive(_handle)
    if _handle.restarts != _restarts_before:
        # The base_url is unchanged, so the client cache would hand back a pool of connections to
        # a process that no longer exists -- a burst of connection errors naming nothing.
        print(f"[Run_Eval] {_name} was restarted -- dropped {reset_client_cache()} cached client(s)")

results = scoring.run_async(scoring.run_scoring(
    plan,
    binding=JUDGE,
    rep=REP,
    concurrency=CONCURRENCY,
    dry_run=DRY_RUN,
    progress=True,
    max_tokens=MAX_TOKENS,
    max_retries=MAX_RETRIES,
    request_timeout=REQUEST_TIMEOUT,
    min_success_ratio=MIN_SUCCESS_RATIO,
))

if results.empty:
    print("nothing was scored -- the plan was empty (a re-run costs zero calls by design).")
else:
    print("\n" + results["status"].value_counts().to_string())

    _written = results[results["status"] == "written"]
    if not _written.empty:
        print(f"\nwrote {len(_written)} partition(s), {int(_written['n_rows'].sum()):,} rows, "
              f"in {_written['elapsed_s'].sum() / 60.0:.1f} min")

    _errors = results[results["status"] == "error"]
    if not _errors.empty:
        print(f"\n{len(_errors)} partition(s) FAILED. They stay in the plan; re-run the plan cell "
              f"and this one to retry only these:")
        display(_errors[["arm_label", "model_state", "metric", "error"]])

## 7. Verify

Re-discover the work list: after a successful run it must be **empty**, which is the same test that
makes a second `Run_Eval` cost zero calls. Then read the lake back through the normal analysis
loader and check the shape every downstream family assumes.

Two things worth looking at in the coverage table:

- **personas per scored state.** 96 is a complete state. Fewer means fewer conversations were on
  disk when it was scored -- the partition is complete *for what was there*, and the state re-enters
  the plan (and is fully re-scored) if the missing conversations appear later. On the Google Drive
  symlink, "the directory reads as empty" is **not** proof the conversations are missing: the mount
  can wedge on a single folder and report zero entries while every file is present in Drive. Check
  the cloud before regenerating anything.
- **ungraded rows.** A conversation the judge could not grade is written as a NaN row with
  `oracle_success=False`, on purpose: a visible hole beats an absent one, which would make the
  partition permanently incomplete and hide the fact that it could not be graded at all.

In [ ]:
remaining = scoring.discover_scorable(ARMS, judge=JUDGE_TAG, rep=REP, metrics=METRIC_KEYS)

if DRY_RUN:
    print("DRY_RUN was on -- no calls were made, so the plan is unchanged.")
elif not remaining.empty:
    display(remaining[["arm_label", "model_state", "metric", "n_conversations", "n_rows_existing"]])
    raise AssertionError(
        f"{len(remaining)} partition(s) are still missing after the run. Re-running the plan cell "
        f"and the run cell picks up exactly these -- a completed partition is skipped, so the retry "
        f"costs only what failed. Read the 'error' column in the run cell first."
    )
else:
    print(f"PLAN IS EMPTY: every requested partition exists for judge={JUDGE_TAG}, rep={REP}. "
          f"A re-run of this notebook would now make zero grading calls.")

scores = data.load_scores_long(
    ARMS, METRIC_KEYS, judge=JUDGE_TAG, rep=REP, attach_persona=False, cache=False,
)

if scores.empty:
    print("\nno scores on disk for this judge/rep -- nothing to summarise.")
else:
    _frame = scores.copy()
    _frame["graded"] = (_frame["oracle_success"].fillna(False).astype(bool)
                        if "oracle_success" in _frame.columns else _frame["score"].notna())

    coverage = (_frame.groupby(["arm_label", "model_state", "metric"], as_index=False)
                      .agg(rows=("persona_id", "size"),
                           personas=("persona_id", "nunique"),
                           graded=("graded", "sum"),
                           mean_score=("score", "mean")))
    coverage["ungraded"] = coverage["rows"] - coverage["graded"]

    per_arm = (coverage.groupby("arm_label", as_index=False)
                       .agg(states=("model_state", "nunique"),
                            partitions=("metric", "size"),
                            min_personas=("personas", "min"),
                            max_personas=("personas", "max"),
                            ungraded_rows=("ungraded", "sum")))
    print(f"\nPer-arm coverage (judge={JUDGE_TAG}, rep={REP}; a complete state is "
          f"{len(METRIC_KEYS)} metrics x {N_PERSONAS} personas):")
    display(per_arm)

    _short = coverage[coverage["personas"] != N_PERSONAS]
    if _short.empty:
        print(f"OK: every scored state carries {N_PERSONAS} personas on every metric.")
    else:
        print(f"NOTE: {len(_short)} partition(s) hold fewer than {N_PERSONAS} personas -- that is "
              f"how many conversations were on disk, not a scoring failure. If conversations are "
              f"missing, check the Drive mount before regenerating them.")
        display(_short)

    _ungraded = int(coverage["ungraded"].sum())
    if _ungraded:
        print(f"NOTE: {_ungraded} row(s) across the lake are NaN (oracle_success=False) -- visible "
              f"holes, not absences. A rising count is the early warning for a grader drifting "
              f"off-schema; re-run the sanity gate if it grows.")

## 8. Prompt-length gate -- the Phase 2 measurement

`SERVE_MAX_MODEL_LEN = 16384` was sized on **Exp3** transcripts written by a `gpt-4o-mini`
patient (full Q2 prompts: median 3,794 / p95 6,524 / max 10,042 tokens on o200k; 10,279 on the
Llama-3.2 tokenizer), with ~60% headroom. Exp4's Gemma patient may write longer turns. A prompt
over the cap is a conversation that **cannot be graded**, the ones that overflow are the LONGEST
-- whose length varies by arm and by K -- so an overflow is arm-dependent biased missingness on
the headline metric, and it is silent: an unscoreable conversation is simply absent. This cell
re-takes the measurement on the real Exp4 conversations that were just graded, in the judge's
own tokenizer, through `tools.oracle_sanity.prompt_length_report`.

**Hard gate:** zero prompts over the **server's** `max_model_len` -- read back from the
server's `/tokenize` answer, not from the literal above: an adopted server keeps the cap it
was launched with, so `SERVE_MAX_MODEL_LEN` is only what *this* notebook launches (and the
fallback cap when no server is used at all). If the two differ the cell says so.
**Note only:** headroom under 1.25x.
If it fails, raise `SERVE_MAX_MODEL_LEN` here **and** `VLLM_MAX_MODEL_LEN` in both trainer
notebooks, restart the server (an adopted server keeps its old cap), and re-score the affected
partitions. **Never lower the cap to save memory** -- give memory back via
`SERVE_GPU_MEMORY_UTILIZATION`.

> The gate is vacuous on an empty lake (nothing to measure, and it says so) and REFUSES to pass
> when the tools layer has no `prompt_length_report` -- an undecidable gate must not read as a
> green one.

In [ ]:
import json

if not PROMPT_LENGTH_GATE:
    print("PROMPT_LENGTH_GATE is off -- the Phase 2 prompt-length measurement was NOT taken. "
          "Record that in the run notes: the 16384 cap then stands on Exp3 data only.")
else:
    transcripts = scoring.gather_transcripts(ARMS)
    _n_arms = 0 if transcripts.empty else transcripts["experiment_name"].nunique()
    print(f"{len(transcripts)} conversation(s) on disk across {_n_arms} arm(s)")

    length_report = scoring.prompt_length_gate(
        transcripts,
        # The cap is the SERVER's when one is used (/tokenize reports it): an adopted server
        # keeps its launch-time --max-model-len, so the literal must not override it. Only
        # without a server (vendor judge / offline estimate) is the literal the fallback cap.
        max_model_len=None if JUDGE.base_url else SERVE_MAX_MODEL_LEN,
        questionnaire_ids=PROMPT_LENGTH_QUESTIONNAIRES,
        model=JUDGE.model,               # count in the served judge's own tokenizer ...
        base_url=JUDGE.base_url,         # ... through the server's /tokenize (THE measurement)
    )
    _cap = length_report.get("max_model_len")
    _cap_source = length_report.get("max_model_len_source", "?")
    if JUDGE.base_url and _cap is not None and int(_cap) != int(SERVE_MAX_MODEL_LEN):
        print(f"NOTE: the server's max_model_len is {_cap} ({_cap_source}); SERVE_MAX_MODEL_LEN "
              f"here is {SERVE_MAX_MODEL_LEN}. The SERVED cap is the one measured against -- an "
              f"adopted server keeps its launch-time cap. Keep the two in step.")
    _format = getattr(oracle_sanity, "format_prompt_length_report", None)
    print(_format(length_report) if _format
          else json.dumps(length_report, indent=2, default=str))

    _ok, _messages = scoring.check_prompt_length_gate(length_report)
    for _m in _messages:
        print("  " + _m)
    if not _ok:
        raise RuntimeError(
            "PROMPT-LENGTH GATE FAILED -- some conversations cannot be graded at the served "
            f"max_model_len={_cap} ({_cap_source}). Raise SERVE_MAX_MODEL_LEN here and "
            "VLLM_MAX_MODEL_LEN in both trainer notebooks above the longest prompt, restart "
            "the server (an adopted server keeps its old cap), and re-run from the plan cell."
        )
    if int(length_report.get("n_transcripts") or 0) == 0:
        print("\nPROMPT-LENGTH GATE: nothing measured (vacuous pass) -- no conversations on disk. "
              "The cap still stands on Exp3 data only; re-run this cell once an arm has landed.")
    else:
        print(f"\nPROMPT-LENGTH GATE PASSED -- every measured oracle prompt fits the served "
              f"max_model_len={_cap} ({_cap_source}).")


## What next

The lake is written. From here:

```powershell
cd Exp4_OpenStack\eda
..\..\.venv\Scripts\python.exe tools\render_results.py       # every family
```

or open a family notebook directly -- `notebooks/arms/outcomes.ipynb`,
`notebooks/lookahead/reward.ipynb`, `notebooks/method/contrast.ipynb`,
`notebooks/compute/cost.ipynb`. They auto-discover arms and read this lake; none of them scores
anything.

**Again: this notebook wrote nothing under `results/`.** Its output is entirely
`data/eval_scores/judge=<tag>/rep=<r>/metric=<M>/<EXPERIMENT_NAME>/model_iter_<N>.parquet`.

If the prompt-length gate above raised, the cap moves in THREE places -- `SERVE_MAX_MODEL_LEN`
here and `VLLM_MAX_MODEL_LEN` in both trainer notebooks -- and the spec's VRAM budget note
(CLAUDE.md) is updated with the new measured maximum.

To force a clean re-score with the local judge, delete the partition directory and re-run this
notebook -- it costs GPU time and no money. To add a second grader, change `JUDGE_MODEL` and run
again: the scores land in that grader's own `judge=<tag>/` partition and every family can then put
the two side by side.

> **Never average raw scores across judges.** One grader shares a model with the TRAINING oracle --
> the thing the policy was optimized against -- and any other is held out. That is train-vs-test,
> not two raters of one construct, and they do not share a scale. Combine only contrasts (a
> difference between two model states under *one* judge) or standardized quantities.